In [0]:
%pip install -U langchain langchain-community databricks-langchain langchain_chroma pypdf
 
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [0]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# 1. CONFIG
# ============================================================

RESUME_PATH = "/Volumes/dev/bronze/raw/resumes/"

CATALOG = "dev"
SCHEMA = "bronze"
TABLE = f"{CATALOG}.{SCHEMA}.resume_chunks"


# ============================================================
# 2. FIND PDFs
# ============================================================

pdf_files = list(Path(RESUME_PATH).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files")

for pdf in pdf_files:
    print(" -", pdf.name)


# ============================================================
# 3. LOAD PDFs
# ============================================================

documents = []

for pdf_file in pdf_files:

    print(f"\nLoading: {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))

    docs = loader.load()

    for doc in docs:

        doc.metadata["source_file"] = pdf_file.name

        # Temporary candidate identifier.
        # Later we can extract the actual candidate name.
        doc.metadata["candidate_id"] = pdf_file.stem

    documents.extend(docs)


print(f"\nTotal pages loaded: {len(documents)}")


# ============================================================
# 4. CHUNK DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")


# ============================================================
# 5. CREATE DATAFRAME
# ============================================================

from pyspark.sql import Row

rows = []

for i, chunk in enumerate(chunks):

    rows.append(
        Row(
            chunk_id=i,
            candidate_id=chunk.metadata.get("candidate_id"),
            source_file=chunk.metadata.get("source_file"),
            page_number=chunk.metadata.get("page", 0),
            content=chunk.page_content
        )
    )


df = spark.createDataFrame(rows)


# ============================================================
# 6. WRITE TO DELTA TABLE
# ============================================================

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)


print(f"\nDelta table created: {TABLE}")


# ============================================================
# 7. VERIFY
# ============================================================

display(
    spark.table(TABLE)
    .orderBy("candidate_id", "page_number", "chunk_id")
)

In [0]:
%pip install -U databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "dev"
SCHEMA = "bronze"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.resume_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.resume_chunks_index"

AI_SEARCH_ENDPOINT = "vector_db"

# Databricks embedding model
EMBEDDING_MODEL = "databricks-gte-large-en"


# ============================================================
# 1. CREATE AI SEARCH CLIENT
# ============================================================

client = AISearchClient()

print("AI Search client created")

In [0]:
spark.sql(f"""
ALTER TABLE {SOURCE_TABLE}
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

print("Change Data Feed enabled")

In [0]:
index = client.create_delta_sync_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="content",
    embedding_model_endpoint_name=EMBEDDING_MODEL
)

print("AI Search index creation started")
print(index.describe())

In [0]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient(disable_notice=True)

AI_SEARCH_ENDPOINT = "vector_db"

INDEX_NAME = "dev.bronze.resume_index"

index = client.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

print(index.describe())

In [0]:
import time

while True:

    status = index.describe()

    print(status["status"])

    if status["status"]["detailed_state"].startswith("ONLINE"):
        print("\n✅ AI Search index is ONLINE")
        break

    print("Waiting for AI Search index...")
    time.sleep(10)

In [0]:
query = "Who has Databricks experience?"

results = index.similarity_search(
    query_text=query,
    columns=[
        "chunk_id",
        "candidate_id",
        "source_file",
        "page_number",
        "content"
    ],
    num_results=5
)

results

In [0]:
for row in results["result"]["data_array"]:

    print("=" * 80)

    print("Candidate:", row[1])
    print("Source:", row[2])
    print("Page:", row[3])

    print("\nContent:")
    print(row[4])